# Gold Layer Analytics

Objective:
Generate business-ready KPIs and analytical tables from Silver Layer data.

Outputs:
1. Customer Summary
2. Branch Summary
3. Loan Summary
4. Transaction Summary

In [0]:
customers_silver = spark.table(
    "banking_catalog.silver.customers"
)

accounts_silver = spark.table(
    "banking_catalog.silver.accounts"
)

customer_summary = (
    customers_silver.join(
        accounts_silver,
        "customer_id",
        "inner"
    )
)

now we are strarting to Create Customer KPI

In [0]:
from pyspark.sql.functions import sum

customer_summary = (
    customer_summary
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        sum("balance")
        .alias("total_balance")
    )
)

In [0]:
display(customer_summary)

customer_id,customer_name,total_balance
C104,Priya Singh,90000
C101,Rahul Sharma,50000
C103,Neha Gupta,60000
C105,Arjun Kumar,120000
C102,Amit Verma,75000


In [0]:
from pyspark.sql.functions import desc

customer_summary = (
    customer_summary
    .orderBy(desc("total_balance"))
)

display(customer_summary)

customer_id,customer_name,total_balance
C105,Arjun Kumar,120000
C104,Priya Singh,90000
C102,Amit Verma,75000
C103,Neha Gupta,60000
C101,Rahul Sharma,50000


In [0]:
#saving the gold layer

customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.customer_summary"
    )

In [0]:
display(customer_summary)

customer_id,customer_name,total_balance
C105,Arjun Kumar,120000
C104,Priya Singh,90000
C102,Amit Verma,75000
C103,Neha Gupta,60000
C101,Rahul Sharma,50000


Table 2 - Branch Summary
Business Question

AS a Bank Manager.

I want to know:

Which branch has the highest deposits?

so i  don't want millions of account records.

I want a summary like:

branch	total_balance
Delhi	350000
Mumbai	250000
Noida	180000

This is exactly what Gold Layer does.

In [0]:
display(customers_silver)

customer_id,customer_name,age,city
C103,Neha Gupta,27,Pune
C105,Arjun Kumar,40,Chennai
C104,Priya Singh,31,Bangalore
C101,Rahul Sharma,29,Delhi
C102,Amit Verma,35,Mumbai


In [0]:
accounts_silver.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- balance: integer (nullable = true)



In [0]:
%python
# -- Create Branch Summary Add a new cell below Customer Summary:

from pyspark.sql.functions import sum, desc

branch_summary = (
    accounts_silver
    .groupBy("branch")
    .agg(
        sum("balance").alias("total_balance")
    )
    .orderBy(desc("total_balance"))
)

display(branch_summary)

branch,total_balance
Chennai,120000
Bangalore,90000
Mumbai,75000
Pune,60000
Delhi,50000


In [0]:
#saving the gold branch summary
branch_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.branch_summary"
    )

Loan Summary : How much loan amount has the bank issued?

A bank manager doesn't want to see individual loans.

They want:

Home Loan  → Total Amount

Car Loan   → Total Amount

Personal Loan → Total Amount

This is a classic Gold Layer KPI.

In [0]:
loans_silver = spark.table(
    "banking_catalog.silver.loans"
)

loans_silver.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_amount: integer (nullable = true)
 |-- loan_type: string (nullable = true)



Loan Summary

How much money has the bank lent for each loan type?

Management wants:

loan_type	total_loan_amount
Home Loan	500000
Car Loan	250000
Personal Loan	150000

instead of seeing individual loan records.

In [0]:
from pyspark.sql.functions import sum, desc

loan_summary = (
    loans_silver
    .groupBy("loan_type")
    .agg(
        sum("loan_amount")
        .alias("total_loan_amount")
    )
    .orderBy(desc("total_loan_amount"))
)

display(loan_summary)

loan_type,total_loan_amount
Home,800000
Personal,500000
Car,300000


In [0]:
#saving gold table
loan_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.loan_summary"
    )

Table 4 - Transaction Summary

How much money flows through the bank?

Business users want answers like:

transaction_type	total_amount
Credit	500000
Debit	300000

instead of looking at every transaction.

In [0]:
#first check the transactions table
transactions_silver = spark.table(
    "banking_catalog.silver.transactions"
)

transactions_silver.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant: string (nullable = true)



In [0]:
#first we need to start with transaction summary
from pyspark.sql.functions import sum, desc

transaction_summary = (
    transactions_silver
    .groupBy("transaction_type")
    .agg(
        sum("amount")
        .alias("total_transaction_amount")
    )
    .orderBy(desc("total_transaction_amount"))
)

display(transaction_summary)

transaction_type,total_transaction_amount
Debit,129500
Credit,40000


In [0]:
#saving the gold table
transaction_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.gold.transaction_summary"
    )

Fraud Detection Analytics
What is Fraud Detection?
Simple Explanation

Banks process millions of transactions daily.

Some transactions may be suspicious:

Customer normally spends ₹2,000

Suddenly spends ₹2,00,000

⚠ Suspicious

or

10 transactions in 1 minute

⚠ Suspicious

Fraud Detection systems help identify these cases.rf

Transactions
      │
      ▼

Fraud Rules Engine
      │
      ▼

Fraud Alerts Table
      │
      ▼

Investigation Team

In [0]:
transactions_silver.select(
    "merchant"
).distinct().show()

In [0]:
transactions_silver.select(
    "merchant"
).distinct().show()

+---------+
| merchant|
+---------+
|CarDealer|
| Flipkart|
|   Amazon|
|    Apple|
|    Bonus|
|   Swiggy|
|   Salary|
+---------+



First Important Banking Lesson

Fraud detection is not about finding fraud with 100% certainty.

It's about:

Finding suspicious transactions

which need investigation.